# Project Part 1: Movie Data Collection

---

**Course:** Data Mining  
**Students:**  
| Name | ID |
|------|----|
| Itamar Rot | 211994025 |
| Nir Avrahamoff | 209413053 |

---

## Workflow Overview
This notebook builds a comprehensive movie dataset in four main steps:

1. **Data Loading** — Downloading raw files from IMDb's public datasets
2. **Filtering** — Keeping only movies starting with Q or R, released by 2024, and 60–300 minutes long
3. **Enrichment** — Filling in missing details (budget, revenue, language, plot) via Wikipedia and OMDb API
4. **Cleaning & Export** — Standardizing financial data to USD millions and exporting the final dataset

<div align="center">
<img src="https://raw.githubusercontent.com/NirAvrahamoff/movie-dataset-mining/f235dbdd664a70f89231b2a36ee4c4092eb62854/pipeline.png" width="900">
</div>

## Step 1: Necessary Libraries
> Importing all required packages before we begin.

| Library | Purpose |
|---------|---------|
| `pandas` | Table operations and data management |
| `requests` / `BeautifulSoup` | Web scraping Wikipedia pages |
| `re` | Cleaning currency symbols and footnotes |
| `time` | Adding delays to avoid server blocks |
| `urllib.parse` | Handling special characters in URLs |

In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
import urllib.parse

## Step 2: IMDb Data Sources

> Defining the URLs for the official IMDb non-commercial datasets.

| File | Contents |
|------|----------|
| `title.basics.tsv.gz` | Movie titles, release year, runtime, genres |
| `title.ratings.tsv.gz` | Average rating and vote count per movie |
| `title.principals.tsv.gz` | Cast and crew for every title |
| `name.basics.tsv.gz` | Actor names linked to their IMDb ID |

In [3]:
# IMDb non-commercial datasets URLs
url_basics = 'https://datasets.imdbws.com/title.basics.tsv.gz'
url_ratings = 'https://datasets.imdbws.com/title.ratings.tsv.gz'
url_principals = 'https://datasets.imdbws.com/title.principals.tsv.gz'
url_names = 'https://datasets.imdbws.com/name.basics.tsv.gz'

## Step 3: Initial Filtering

> Loading the basics file and narrowing it down to our target movies.

| Filter | Condition | Reason |
|--------|-----------|--------|
| `titleType` | `== 'movie'` | Exclude TV shows, shorts, etc. |
| `startYear` | `<= 2024` | No unreleased or future movies |
| `runtimeMinutes` | `60 – 300` | Feature-length films only |
| `primaryTitle` | Starts with **Q** or **R** | Assignment requirement |

After all filters: **13,329 movies** remain.

In [4]:
# Load title.basics table
df_basics = pd.read_csv(url_basics, sep='\t', na_values='\\N', compression='gzip', low_memory=False)

# Filter 1: Keep only movies
movies = df_basics[df_basics['titleType'] == 'movie'].copy()

# Filter 2: Release year up to 2024
movies['startYear'] = pd.to_numeric(movies['startYear'], errors='coerce')
movies = movies[movies['startYear'] <= 2024]

# Filter 3: Runtime between 60 and 300 minutes
movies['runtimeMinutes'] = pd.to_numeric(movies['runtimeMinutes'], errors='coerce')
movies = movies[(movies['runtimeMinutes'] >= 60) & (movies['runtimeMinutes'] <= 300)]

# Filter 4: Movies starting with 'Q' or 'R'
movies = movies[movies['primaryTitle'].str.match(r'^[QRqr]', na=False)]

# Convert genres to a list of strings
movies['genres'] = movies['genres'].str.split(',')

movies = movies[['tconst', 'primaryTitle', 'startYear', 'runtimeMinutes', 'genres']]

## Step 4: Ratings and Cast Management

> Adding ratings data and extracting the top 5 lead actors per movie.

**Why process `title.principals` in chunks?**  
This file is several gigabytes. Loading it all at once would crash most machines.  
Instead, we read it in batches of **500,000 rows**, filter each batch immediately to keep only our Q/R movies, and discard the rest — so we only hold a small fraction of the data in memory at any time.

**Logic summary:**
1. Merge IMDb ratings (`averageRating`, `numVotes`) into the movies table
2. Read `title.principals` in chunks → keep only rows matching our movie IDs
3. Filter for `actor` and `actress` categories, sort by `ordering`, take top 5 per movie
4. Sort all movies by `numVotes` (descending) → keep the **top 5,000**

In [5]:
# Load and merge ratings
df_ratings = pd.read_csv(url_ratings, sep='\t', na_values='\\N', compression='gzip', low_memory=False)
movies = pd.merge(movies, df_ratings, on='tconst', how='left')

# Load names to get lead actors
df_names = pd.read_csv(url_names, sep='\t', na_values='\\N', compression='gzip', usecols=['nconst', 'primaryName'])

# ---------------------------------------------------------
# THE FIX: Reading the massive 'principals' file in chunks!
# ---------------------------------------------------------
print("Loading actors efficiently using chunks to prevent MemoryError...")
top_tconsts = set(movies['tconst']) # Create a fast lookup set of the movie IDs we actually care about
chunks = pd.read_csv(url_principals, sep='\t', na_values='\\N', compression='gzip', usecols=['tconst', 'ordering', 'nconst', 'category'], chunksize=500000)

principals_list = []
for chunk in chunks:
    # Filter the chunk IMMEDIATELY to keep only the movies we need, throwing away 99% of the data
    filtered_chunk = chunk[chunk['tconst'].isin(top_tconsts)]
    if not filtered_chunk.empty:
        principals_list.append(filtered_chunk)

# Combine only the small filtered chunks back together
df_principals = pd.concat(principals_list)
# ---------------------------------------------------------

# Filter for actors and actresses
actors = df_principals[df_principals['category'].isin(['actor', 'actress'])]

# Sort by movie and ordering, then remove duplicate actors per movie
actors = actors.sort_values(['tconst', 'ordering'])
actors = actors.drop_duplicates(subset=['tconst', 'nconst'])

# Group by movie to get the top 5 actor IDs (nconst) as a list
lead_actors_list = actors.groupby('tconst').head(5) \
                         .groupby('tconst')['nconst'].apply(list).reset_index()

# Rename column to match the assignment instructions perfectly
lead_actors_list.rename(columns={'nconst': 'lead_actors_ids'}, inplace=True)

# Merge back to the main movies dataframe and sort by popularity
movies = pd.merge(movies, lead_actors_list, on='tconst', how='left')
movies = movies.sort_values(by='numVotes', ascending=False)
print(f"Total movies prepared: {len(movies)}")

Loading actors efficiently using chunks to prevent MemoryError...
Total movies prepared: 13332


## Step 5: Web Scraping & API Functions

> Defining the helper functions used to enrich our dataset.

| Function | Source | What it does |
|----------|--------|--------------|
| `clean_wiki_text()` | — | Removes footnotes like `[1]`, fixes line breaks |
| `get_wikipedia_data()` | Wikipedia | Scrapes `Language`, `Country`, `budget`, `BoxOffice`, `plot` from the movie's infobox. Validates the year (±1) to avoid matching the wrong film. Skips disambiguation pages automatically. |
| `get_omdb_data()` | OMDb API | Fetches `Language`, `Country`, and `plot` using the movie's IMDb ID (`tconst`) |


In [6]:
# Set your OMDb API Key here
OMDB_API_KEY = "b654e7ae"

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def clean_wiki_text(text):
    if not text:
        return None
    cleaned = re.sub(r'\[.*?\]', '', text)
    cleaned = cleaned.replace('\n', ', ') 
    cleaned = re.sub(r',\s*,', ',', cleaned)
    return cleaned.strip(' ,')

def get_wikipedia_data(movie_title, expected_year, fetch_only_financials=False):
    year_str = str(int(float(expected_year))) if pd.notna(expected_year) else ""
    encoded_title = urllib.parse.quote(movie_title.replace(' ', '_'))
    
    urls = [
        f"https://en.wikipedia.org/wiki/{encoded_title}_({year_str}_film)" if year_str else "",
        f"https://en.wikipedia.org/wiki/{encoded_title}_(film)",
        f"https://en.wikipedia.org/wiki/{encoded_title}"
    ]
    urls = [url for url in urls if url]
    
    for url in urls:
        try:
            response = requests.get(url, headers=HEADERS, timeout=5)
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')
                
                if soup.find(class_="disambigbox"):
                    continue 

                infobox = soup.find('table', class_=lambda c: c and 'infobox' in c)
                
                if infobox:
                    infobox_text = clean_wiki_text(infobox.text)
                    try:
                        year_int = int(year_str)
                        if not (str(year_int) in infobox_text or str(year_int - 1) in infobox_text or str(year_int + 1) in infobox_text):
                            continue 
                    except ValueError:
                        pass
                
                data = {'Language': None, 'Country': None, 'budget': None, 'BoxOffice': None, 'plot': None}
                if infobox:
                    for row in infobox.find_all('tr'):
                        th, td = row.find('th'), row.find('td')
                        if th and td:
                            header = th.text.strip().lower().replace('\xa0', ' ')
                            raw_text = td.get_text(separator=', ', strip=True)
                            clean_val = clean_wiki_text(raw_text)
                            
                            if not fetch_only_financials:
                                if 'language' in header: data['Language'] = clean_val
                                elif 'countr' in header or 'origin' in header: data['Country'] = clean_val
                            
                            if 'budget' in header: data['budget'] = clean_val
                            elif 'box office' in header: data['BoxOffice'] = clean_val
                
                if not fetch_only_financials:
                    plot_heading = soup.find(id=lambda x: x and x.lower() in ['plot', 'synopsis', 'premise', 'story'])
                    if plot_heading and plot_heading.parent:
                        plot_paragraphs = []
                        curr_elem = plot_heading.parent.find_next_sibling()
                        
                        while curr_elem and curr_elem.name not in ['h2', 'h3']:
                            if curr_elem.name == 'p' and curr_elem.text.strip():
                                plot_paragraphs.append(clean_wiki_text(curr_elem.text))
                            curr_elem = curr_elem.find_next_sibling()
                            
                        if plot_paragraphs:
                            data['plot'] = " ".join(plot_paragraphs)
                
                return data
        except: continue
    return {'Language': None, 'Country': None, 'budget': None, 'BoxOffice': None, 'plot': None}

def get_omdb_data(tconst):
    try:
        r = requests.get(f"http://www.omdbapi.com/?i={tconst}&apikey={OMDB_API_KEY}", timeout=5)
        return r.json() if r.status_code == 200 else None
    except: return None

## Step 6: The Collection Loop

> Main loop that collects enrichment data for all 5,000 movies.

**Source priority logic:**

| Movies | `Language` / `Country` / `plot` | `budget` / `BoxOffice` |
|--------|---------------------------------|------------------------|
| 1 – 1,000 | OMDb API (fallback: Wikipedia) | Wikipedia always |
| 1,001 – 5,000 | Wikipedia | Wikipedia |

> **Why Wikipedia for financials even when OMDb is available?**  
> OMDb frequently returns `N/A` for budget and box office. Wikipedia's infobox is far more complete and includes the original currency, ranges, and updated figures.

**Checkpoints:** A backup file (`dataset_raw_checkpoint.csv`) is saved automatically every 500 movies to prevent data loss if the connection drops during the multi-hour run.


In [7]:
valid_movies_data = []
omdb_calls = 0

print("Starting data collection for 5000 movies...")

for index, row in movies.head(5000).iterrows():
    combined_row = row.to_dict()
    
    # 1. Try OMDb first (up to 1000 calls)
    if omdb_calls < 1000:
        omdb_json = get_omdb_data(row['tconst'])
        omdb_calls += 1
        
        # We fetch financials from Wikipedia anyway, because OMDb lacks reliable budget/boxoffice formats
        wiki_financials = get_wikipedia_data(row['primaryTitle'], row['startYear'], fetch_only_financials=True)
        combined_row.update(wiki_financials)
        
        if omdb_json and omdb_json.get('Response') == 'True':
            combined_row['Language'] = omdb_json.get('Language') if omdb_json.get('Language') != 'N/A' else None
            combined_row['Country'] = omdb_json.get('Country') if omdb_json.get('Country') != 'N/A' else None
            combined_row['plot'] = omdb_json.get('Plot') if omdb_json.get('Plot') != 'N/A' else None
        else:
            # If OMDb fails, fallback entirely to Wikipedia
            wiki_full_data = get_wikipedia_data(row['primaryTitle'], row['startYear'])
            combined_row.update(wiki_full_data)
            
    # 2. Beyond 1000 calls: Use ONLY Wikipedia
    else:
        wiki_data = get_wikipedia_data(row['primaryTitle'], row['startYear'])
        combined_row.update(wiki_data)

    valid_movies_data.append(combined_row)
    
    # Progress update every 100 movies
    if len(valid_movies_data) % 100 == 0:
        print(f"Collected: {len(valid_movies_data)}/5000 | OMDb Calls: {omdb_calls}")
        
        # Checkpoint: Auto-save every 500 movies to prevent data loss
        if len(valid_movies_data) % 500 == 0:
            pd.DataFrame(valid_movies_data).to_csv('dataset_raw_checkpoint.csv', index=False, encoding='utf-8-sig')

    time.sleep(0.1)

# Final raw save with UTF-8 encoding for Excel compatibility
pd.DataFrame(valid_movies_data).to_csv('dataset_raw.csv', index=False, encoding='utf-8-sig')
print("Collection complete.")

Starting data collection for 5000 movies...
Collected: 100/5000 | OMDb Calls: 100
Collected: 200/5000 | OMDb Calls: 200
Collected: 300/5000 | OMDb Calls: 300
Collected: 400/5000 | OMDb Calls: 400
Collected: 500/5000 | OMDb Calls: 500
Collected: 600/5000 | OMDb Calls: 600
Collected: 700/5000 | OMDb Calls: 700
Collected: 800/5000 | OMDb Calls: 800
Collected: 900/5000 | OMDb Calls: 900
Collected: 1000/5000 | OMDb Calls: 1000
Collected: 1100/5000 | OMDb Calls: 1000
Collected: 1200/5000 | OMDb Calls: 1000
Collected: 1300/5000 | OMDb Calls: 1000
Collected: 1400/5000 | OMDb Calls: 1000
Collected: 1500/5000 | OMDb Calls: 1000
Collected: 1600/5000 | OMDb Calls: 1000
Collected: 1700/5000 | OMDb Calls: 1000
Collected: 1800/5000 | OMDb Calls: 1000
Collected: 1900/5000 | OMDb Calls: 1000
Collected: 2000/5000 | OMDb Calls: 1000
Collected: 2100/5000 | OMDb Calls: 1000
Collected: 2200/5000 | OMDb Calls: 1000
Collected: 2300/5000 | OMDb Calls: 1000
Collected: 2400/5000 | OMDb Calls: 1000
Collected: 250

## Step 7: Financial Cleaning & Export

> Standardizing all financial data and exporting the final dataset.

The raw financial values scraped from Wikipedia are messy. The `clean_financial_data()` function handles:

| Challenge | Example | Solution |
|-----------|---------|----------|
| Multiple currencies | `£5 million`, `₹200 crore` | Convert to USD using exchange rates |
| Budget ranges | `$1–2 million` | Take the average |
| Scale words | `billion`, `crore`, `lakh` | Convert to a single number |
| Inflation notes | `($10M in 2024 dollars)` | Strip parenthetical text |


In [8]:
def clean_financial_data(val):
    if pd.isna(val) or val is None: 
        return None
        
    # Convert to lowercase and remove spaces/commas to facilitate extraction
    s = str(val).lower().replace(',', '').replace('\xa0', ' ')
    
    # Remove secondary conversions or inflation adjustments often enclosed in parentheses
    s = re.sub(r'\(.*?\)', '', s)
    
    # ULTIMATE EXCHANGE RATES DICTIONARY 
    # Covers major and minor global currencies prevalent in international cinema
    exchange_rates = {
        # Majors
        '£': 1.36, 'gbp': 1.36,                          # British Pound
        '€': 1.17, 'eur': 1.17,                          # Euro
        '₪': 0.34, 'ils': 0.34, 'shekel': 0.34,          # Israeli New Shekel
        
        # Asia / Pacific
        '₹': 0.012, 'inr': 0.012, 'rs': 0.012, 'rupee': 0.012, # Indian Rupee
        '¥': 0.0065, 'jpy': 0.0065, 'yen': 0.0065,       # Japanese Yen
        'c$': 0.73, 'cad': 0.73,                         # Canadian Dollar
        'a$': 0.65, 'aud': 0.65,                         # Australian Dollar
        'nz$': 0.60, 'nzd': 0.60,                        # New Zealand Dollar
        '₩': 0.00073, 'krw': 0.00073, 'won': 0.00073,    # South Korean Won
        'cn¥': 0.14, 'cny': 0.14, 'rmb': 0.14, 'yuan': 0.14, # Chinese Yuan / RMB
        'hk$': 0.13, 'hkd': 0.13,                        # Hong Kong Dollar
        'nt$': 0.031, 'twd': 0.031,                      # New Taiwan Dollar
        '฿': 0.027, 'thb': 0.027, 'baht': 0.027,         # Thai Baht
        '₱': 0.017, 'php': 0.017, 'peso': 0.017,         # Philippine Peso
        'rp': 0.000062, 'idr': 0.000062, 'rupiah': 0.000062, # Indonesian Rupiah
        'rm': 0.21, 'myr': 0.21, 'ringgit': 0.21,        # Malaysian Ringgit
        
        # Europe (Non-Euro)
        '₽': 0.011, 'rub': 0.011, 'ruble': 0.011,        # Russian Ruble
        'kr': 0.09, 'sek': 0.09, 'dkk': 0.14, 'nok': 0.09, 'krone': 0.09, 'krona': 0.09, # Swedish/Danish/Norwegian Krone
        'chf': 1.10, 'franc': 1.10,                      # Swiss Franc
        'kč': 0.043, 'czk': 0.043, 'koruna': 0.043,      # Czech Koruna
        'zł': 0.25, 'pln': 0.25, 'zloty': 0.25,          # Polish Zloty
        'ft': 0.0028, 'huf': 0.0028, 'forint': 0.0028,   # Hungarian Forint
        '₺': 0.031, 'try': 0.031, 'lira': 0.031,         # Turkish Lira
        
        # Americas & Africa
        'r$': 0.19, 'brl': 0.19, 'real': 0.19,           # Brazilian Real
        'zar': 0.053, 'rand': 0.053                      # South African Rand
    }
    
    rate = 1.0 # Default rate is US Dollar
    for symbol, conversion in exchange_rates.items():
        if symbol in s or f"{symbol} " in s:
            rate = conversion
            break
            
    # Smart fix for ranges: "1.3 to 2 million" becomes "1.3 million to 2 million"
    def fix_range_modifiers(match):
        num1_str = match.group(1)
        num2_str = match.group(2)
        mod = match.group(3)
        if float(num1_str) < 1000:
            return f"{num1_str} {mod} to {num2_str} {mod}"
        else:
            return f"{num1_str} to {num2_str} {mod}"

    pattern = r'(\d+(?:\.\d+)?)\s*(?:-|–|to|and|/)\s*(\d+(?:\.\d+)?)\s*(billion|million|crore|lakh|thousand|m|b|k)\b'
    s = re.sub(pattern, fix_range_modifiers, s)
    
    # Extract numbers with their modifiers
    matches = re.finditer(r'(\d+(?:\.\d+)?)\s*(billion|million|crore|lakh|thousand|m|b|k)?\b', s)
    
    parsed_numbers = []
    for match in matches:
        num_str, mod = match.groups()
        try: 
            num = float(num_str)
        except ValueError: 
            continue
            
        if mod:
            if mod in ['billion', 'b']: num *= 1_000_000_000
            elif mod in ['million', 'm']: num *= 1_000_000
            elif mod == 'crore': num *= 10_000_000
            elif mod == 'lakh': num *= 100_000
            elif mod in ['thousand', 'k']: num *= 1_000
        else:
            if num < 1000 and re.search(r'\bmillion\b', s): num *= 1_000_000
            elif num < 1000 and re.search(r'\bbillion\b', s): num *= 1_000_000_000
            elif num < 1000 and re.search(r'\bcrore\b', s): num *= 10_000_000
            elif num < 1000 and re.search(r'\blakh\b', s): num *= 100_000
        
        # Skip pure years
        if not mod and 1900 <= num <= 2030 and num_str.isdigit(): 
            continue
            
        parsed_numbers.append(num)

    if not parsed_numbers: 
        return None
        
    # Prevent averaging vastly different scales (local currency vs USD fallback)
    if len(parsed_numbers) > 1:
        min_val = min(parsed_numbers)
        max_val = max(parsed_numbers)
        if min_val > 0 and max_val > min_val * 10:
            parsed_numbers = [parsed_numbers[0]]
        
    # Calculate average
    avg_raw = sum(parsed_numbers) / len(parsed_numbers)
    
    # Apply exact currency conversion and normalize to millions
    final_usd_millions = (avg_raw * rate) / 1_000_000
    
    return round(final_usd_millions, 2)

# Load the raw file
final_df = pd.read_csv('dataset_raw.csv')

# Clean financials
final_df['budget'] = final_df['budget'].apply(clean_financial_data)
final_df['BoxOffice'] = final_df['BoxOffice'].apply(clean_financial_data)

# Remove trailing zeroes for integer columns
final_df['startYear'] = final_df['startYear'].astype('Int64')
final_df['runtimeMinutes'] = final_df['runtimeMinutes'].astype('Int64')
final_df['numVotes'] = final_df['numVotes'].astype('Int64')

# Reorder columns
required_columns = [
    'tconst', 'primaryTitle', 'startYear', 'genres', 'lead_actors_ids',
    'runtimeMinutes', 'averageRating', 'Language', 'Country', 
    'numVotes', 'budget', 'BoxOffice', 'plot'
]
final_df = final_df[required_columns]

# Save dataset
final_df.to_csv('dataset.csv', index=False, encoding='utf-8-sig')

print("Data cleaning complete. Ready for submission!")

Data cleaning complete. Ready for submission!


## Step 8: Dataset Validation

> A quick sanity check on the final dataset before submission.

In [10]:
df = pd.read_csv('dataset.csv')
df.head()

,tconst,primaryTitle,startYear,genres,lead_actors_ids,runtimeMinutes,averageRating,Language,Country,numVotes,budget,BoxOffice,plot
0,tt0105236,Reservoir Dogs,1992,"['Crime', 'Thriller']","['nm0000172', 'nm0000619', 'nm0000514', 'nm000...",99,8.3,English,United States,1170435,2.1,2.9,When a simple jewelry heist goes horribly wron...
1,tt0082971,Raiders of the Lost Ark,1981,"['Action', 'Adventure']","['nm0000148', 'nm0000261', 'nm0293550', 'nm072...",115,8.4,"English, German, Spanish, Nepali, Hebrew, Arabic",United States,1109959,20.0,389.9,"In 1936, archaeologist Indiana Jones is tasked..."
2,tt0180093,Requiem for a Dream,2000,['Drama'],"['nm0000995', 'nm0001467', 'nm0000124', 'nm000...",102,8.3,English,United States,973150,4.5,7.4,The drug-induced utopias of four Coney Island ...
3,tt0382932,Ratatouille,2007,"['Adventure', 'Animation', 'Comedy']","['nm0004951', 'nm0738918', 'nm0652663', 'nm000...",111,8.1,"English, French","United States, France",943272,150.0,623.7,A rat who can cook makes an unusual alliance w...
4,tt3748528,Rogue One: A Star Wars Story,2016,"['Action', 'Adventure', 'Sci-Fi']","['nm0428065', 'nm0526019', 'nm0876138', 'nm094...",133,7.8,English,United States,755487,240.1,1059.0,"In a time of conflict, a group of unlikely her..."


In [11]:
df.dtypes

tconst              object
primaryTitle        object
startYear            int64
genres              object
lead_actors_ids     object
runtimeMinutes       int64
averageRating      float64
Language            object
Country             object
numVotes             int64
budget             float64
BoxOffice          float64
plot                object
dtype: object

In [12]:
df.describe().applymap(lambda x: f'{x:,.2f}')

,startYear,runtimeMinutes,averageRating,numVotes,budget,BoxOffice
count,"5,000.00","5,000.00","5,000.00","5,000.00",821.00,"1,036.00"
mean,"2,000.45",100.76,5.92,"8,530.74",15.37,414.06
std,23.28,22.59,1.31,"48,368.85",30.77,"10,537.92"
min,"1,913.00",60.00,1.00,80.00,0.00,0.00
25%,"1,989.00",87.00,5.20,163.75,1.00,0.83
50%,"2,009.00",96.00,6.10,402.50,4.50,3.00
75%,"2,018.00",110.00,6.80,"1,586.50",15.00,14.40
max,"2,024.00",283.00,9.60,"1,170,435.00",300.00,"333,059.00"


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   tconst           5000 non-null   object 
 1   primaryTitle     5000 non-null   object 
 2   startYear        5000 non-null   int64  
 3   genres           4992 non-null   object 
 4   lead_actors_ids  4652 non-null   object 
 5   runtimeMinutes   5000 non-null   int64  
 6   averageRating    5000 non-null   float64
 7   Language         2796 non-null   object 
 8   Country          2818 non-null   object 
 9   numVotes         5000 non-null   int64  
 10  budget           821 non-null    float64
 11  BoxOffice        1036 non-null   float64
 12  plot             2406 non-null   object 
dtypes: float64(3), int64(3), object(7)
memory usage: 507.9+ KB


In [14]:
df.duplicated().sum()

0

In [15]:
# Missing values — count and percentage
missing_report = pd.DataFrame({
    'Missing Count': df.isna().sum(),
    'Missing %': (df.isna().sum() / len(df) * 100).round(2)
})
print(missing_report)

                 Missing Count  Missing %
tconst                       0       0.00
primaryTitle                 0       0.00
startYear                    0       0.00
genres                       8       0.16
lead_actors_ids            348       6.96
runtimeMinutes               0       0.00
averageRating                0       0.00
Language                  2204      44.08
Country                   2182      43.64
numVotes                     0       0.00
budget                    4179      83.58
BoxOffice                 3964      79.28
plot                      2594      51.88
